# Renters (specialty-ltv) v1.5.0 - loss ratio refit validation (cat excluded)

In [ ]:
%env ENV_FOR_DYNACONF = prod%env DYNACONF_GIT_BRANCH = feature/B-2895893%env DYNACONF_GIT_CHECKOUT = feature/B-2895893

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport ltv_helpers.non_spark_helpers as nsfrom specialty_ltv import paths as p%matplotlib inlinepd.options.display.max_rows = 2000pd.options.display.max_columns = 500

In [ ]:
P_SCORED_NEW = p.score_internal_resultsP_SCORED_PRIOR = ("tmx-smsiweb/specialty-ltv/prod/"                  "<PRIOR_RELEASE>/score/score_internal_results/")  # >>> fill inprint(P_SCORED_NEW)

## Fits

In [ ]:
ntr_gt0_fits = [16, 88, 90]spl_fits = {    16: [-0.034158, 0.610550, 0.970097],   # Specialty Auto    32: [-0.013335, 0.366484],             # Mfg Home    71: [-0.043571, 0.625204],             # Renters    72: [ 0.000000, 0.315427],             # Landlord    78: [ 0.000000, 0.493024],             # Condo    88: [ 0.000000, 0.788817, 1.180678],   # PUP    90: [-0.023748, 0.565079, 0.769902],   # Boat}prod_fits = {    16: [-0.02294,  0.6347,  1.0057],    32: [-0.01407,  0.6206],    71: [-0.05045,  0.7259],    72: [-0.003477, 0.6060],    78: [-0.03717,  0.82678],    88: [ 0.0,      0.7571,  1.2079],    90: [-0.02329,  0.7527,  0.9611],}LINE_NAMES = {16: "Specialty Auto", 32: "Mfg Home", 71: "Renters", 72: "Landlord",              78: "Condo", 88: "PUP", 90: "Boat"}

In [ ]:
# Renters is line 71. Note 71 also appears in classic's spl_lr_fits - resolve# which repo scores it.spl_fits = {71: spl_fits[71]}prod_fits = {71: prod_fits[71]}

In [ ]:
MAX_NT6 = 18def eval_fit(nt6, line, fits):    """Mirrors create_slope_intercept_spl_lr_df."""    f = fits[line]    if nt6 < 2 and line in ntr_gt0_fits:        return f[2]    return f[1] + f[0] * (min(nt6, MAX_NT6) / 2)

### Fit delta

NTR=0 is the column to read. Landlord and Condo went flat, so nt0 mirrors therenewal intercept:- Landlord 0.6060 -> 0.3154- Condo    0.8268 -> 0.4930Neither was fit as a new-business estimate.

In [ ]:
rows = []for line in sorted(spl_fits):    for nt6 in [0, 2, 4, 6, 10, 18]:        rows.append({            "drv_line": line, "name": LINE_NAMES[line], "ntr": min(nt6, MAX_NT6) / 2,            "prod": eval_fit(nt6, line, prod_fits),            "new": eval_fit(nt6, line, spl_fits),        })delta = pd.DataFrame(rows)delta["diff"] = delta["new"] - delta["prod"]delta.pivot_table(index=["drv_line", "name"], columns="ntr", values="diff").round(4)

## Load

In [ ]:
COLS = ["drv_line", "state", "premium", "premium_new", "premium_renew",        "loss", "cat", "lr_balance_amt", "loss_ratio_bal_factor",        "loss_ratio_cat", "loss_ratio_x_cat_yr1_target",        "loss_ratio_x_cat_yr2_target", "loss_ratio_x_cat_yr3_target",        "term_length"] + [f"loss_{n}" for n in range(10)]df = ns.read_parquet_s3_to_pandas(P_SCORED_NEW)[COLS]df["drv_line"] = df["drv_line"].astype(int)df = df[df["drv_line"].isin(spl_fits)]df.shape

## Propagation check`loss_n` is indexed by nt6 (ntr = n/2), so `loss_0 == loss_1` from the nt6 < 2branch. The per-policy exposure base is constant across n, so    loss_n / loss_0 == eval_fit(n) / eval_fit(0)is base-free - no premium needed, no circularity.

In [ ]:
chk = []for line, g in df.groupby("drv_line"):    base = g["loss_0"]    for n in range(1, 10):        exp = eval_fit(n, line, spl_fits) / eval_fit(0, line, spl_fits)        obs = (g[f"loss_{n}"] / base).replace([np.inf, -np.inf], np.nan)        chk.append({"drv_line": line, "n": n, "expected": exp,                    "obs_mean": obs.mean(), "max_gap": (obs - exp).abs().max()})chk = pd.DataFrame(chk)chk["pass"] = chk["max_gap"] < 1e-4chk.pivot_table(index="drv_line", columns="n", values="max_gap").round(6)

In [ ]:
# Same test against prod_fits - should FAIL if the new fits landed.chk_prod = []for line, g in df.groupby("drv_line"):    base = g["loss_0"]    for n in [2, 4, 9]:        exp = eval_fit(n, line, prod_fits) / eval_fit(0, line, prod_fits)        obs = (g[f"loss_{n}"] / base).replace([np.inf, -np.inf], np.nan)        chk_prod.append({"drv_line": line, "n": n,                         "max_gap_vs_prod": (obs - exp).abs().max()})pd.DataFrame(chk_prod).pivot_table(    index="drv_line", columns="n", values="max_gap_vs_prod").round(6)

## Cat components`loss_ratio_cat` and `loss_ratio_x_cat_yr1/2/3_target` already exist in theoutput, so cat is carried separately and the ex-cat LR has its own balancetargets. Check whether the refit belongs in whatever config feeds those targets,not only in `spl_lr_fits`.

In [ ]:
df.groupby("drv_line")[    ["loss_ratio_cat", "loss_ratio_x_cat_yr1_target",     "loss_ratio_x_cat_yr2_target", "loss_ratio_x_cat_yr3_target",     "loss_ratio_bal_factor"]].mean().round(4)

## New vs prior release`score_internal_results` is post-balancing, so `lr_total` holding steady provesnothing on its own - the balance step would absorb the cat strip either way.`lr_balance_amt` is the discriminating column.- `d_bal` cancels `d_cat` -> balancing swallowed it, the change never reaches the P&L- `d_bal` flat while `lr_ex_cat` drops -> the change is real

In [ ]:
def lr_summary(path, tag):    d = ns.read_parquet_s3_to_pandas(path)[        ["drv_line", "premium", "loss", "cat", "lr_balance_amt"]]    d["drv_line"] = d["drv_line"].astype(int)    g = d[d["drv_line"].isin(spl_fits)].groupby("drv_line", as_index=False).agg(        premium=("premium", "sum"), loss=("loss", "sum"),        cat=("cat", "sum"), bal=("lr_balance_amt", "sum"))    g["lr_ex_cat"] = g["loss"] / g["premium"]    g["lr_total"] = (g["loss"] + g["cat"]) / g["premium"]    g["cat_share"] = g["cat"] / (g["loss"] + g["cat"])    return gnew = lr_summary(P_SCORED_NEW, "new")old = lr_summary(P_SCORED_PRIOR, "prior")cmp = new.merge(old, on="drv_line", suffixes=("_new", "_old"))for c in ["lr_ex_cat", "lr_total", "premium", "cat", "bal"]:    cmp[f"d_{c}"] = cmp[f"{c}_new"] - cmp[f"{c}_old"]cmp["name"] = cmp["drv_line"].map(LINE_NAMES)cmp[["drv_line", "name", "lr_ex_cat_old", "lr_ex_cat_new", "d_lr_ex_cat",     "lr_total_old", "lr_total_new", "d_lr_total",     "d_cat", "d_bal", "d_premium"]].round(3)

### Dollar impact

In [ ]:
cmp["impact"] = cmp["d_lr_ex_cat"] * cmp["premium_new"]tot_p, tot_i = cmp["premium_new"].sum(), cmp["impact"].sum()print(f"premium {tot_p:,.0f}   impact {tot_i:,.0f}   points {tot_i / tot_p * 100:.1f}")out = cmp[["drv_line", "name", "premium_new", "d_lr_ex_cat", "impact"]].copy()out["pct"] = out["impact"] / tot_iout.sort_values("impact").round(3)

## Open items

In [ ]:
# 1. Cat code '9' - real serial or sentinel? True cats cluster in month x state.#    Run on the CLAIMS extract:#    claims["CATCD"].value_counts(dropna=False)#    pd.crosstab(claims.loc[claims.CATCD == "9", "ACTMO"],#                claims.loc[claims.CATCD == "9", "GEOST"])# 2. PUP zero cat - is CATCD populated-and-blank for line 88, or absent?#    claims.loc[claims.ALINE == "88", "CATCD"].isna().mean()# 3. Exposure window - rerun fits on 2023-2024 only. ACTYR 125 is immature.#    Control, not a decision.# 4. Landlord +0.00733 (p<0.001, $1.6B) suppressed by 'slope > 0 -> flat'.#    Refit without NTR=9: positive only WITH the censored bucket means the rule#    is accidentally right; positive without it means real signal on the largest#    line. Cat removal strips the fattest tail, so the post-cat p-value is more#    credible than any pre-cat one.